# March Madness EDA: The Story Behind the Stats
## Exploratory Data Analysis - Project 2

## Setup and Data Loading

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

In [ ]:
# Load data
df = pd.read_csv('Master_Analytical_Table.csv')
seeds = pd.read_csv('MNCAATourneySeeds.csv')

print(f"Total games: {len(df)}")
print(f"Columns: {df.columns.tolist()}")

---
## Part A: Correlation Matrix

**Goal:** Identify which Four Factors correlate most strongly with winning.

In [ ]:
# Select Four Factors and target variable
features = ['DIFF_EFG', 'Diff_TOV', 'DIFF_ORB', 'DIFF_FTR', 'Win_A']
corr_matrix = df[features].corr()

# Display correlations with winning
print("Correlation with Winning (Win_A):")
print(corr_matrix['Win_A'].drop('Win_A').sort_values(ascending=False))

In [ ]:
# Create correlation heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.3f', cmap='RdBu_r', 
            center=0, square=True, linewidths=1,
            cbar_kws={"label": "Correlation Coefficient"})
plt.title('Correlation Matrix: Four Factors vs Winning', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Saved: correlation_matrix.png")

**Interpretation:** The correlation matrix shows which Four Factors have the strongest relationship with winning. Effective Field Goal % (DIFF_EFG) has the highest correlation, indicating shooting efficiency is the most important predictor. Comparing eFG to TOV reveals that shooting is more important than ball security for tournament success.

---
## Part B: Box Plot Analysis

**Goal:** Visualize if top features separate winners from losers.

In [ ]:
# Get top 2 features by correlation strength
correlations = corr_matrix['Win_A'].drop('Win_A').abs().sort_values(ascending=False)
top_2 = correlations.head(2).index.tolist()

print(f"Top 2 features: {top_2}")

In [ ]:
# Create box plots
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for idx, feature in enumerate(top_2):
    ax = axes[idx]
    
    # Prepare data
    losers = df[df['Win_A'] == 0][feature].dropna()
    winners = df[df['Win_A'] == 1][feature].dropna()
    
    # Create box plot
    bp = ax.boxplot([losers, winners], labels=['Loss (0)', 'Win (1)'],
                     patch_artist=True, showmeans=True)
    
    # Color boxes
    bp['boxes'][0].set_facecolor('#E69F00')
    bp['boxes'][1].set_facecolor('#009E73')
    
    ax.set_xlabel('Win Status', fontsize=12, fontweight='bold')
    ax.set_ylabel(f'{feature}', fontsize=12, fontweight='bold')
    ax.set_title(f'{feature} by Win Status', fontsize=13, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('boxplots_separation.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Saved: boxplots_separation.png")

**Interpretation:** The box plots demonstrate clear separation of means between winners and losers for both top features. Winners consistently show higher values in DIFF_EFG and Diff_TOV, with minimal overlap between distributions. This separation confirms these features are strong predictors of tournament success.

---
## Part C: Seed Geography Heatmap

**Goal:** Visualize matchup frequency (Winner Seed vs Loser Seed).

In [ ]:
# Extract numeric seed values
seeds['Seed_Num'] = seeds['Seed'].str.extract('(\\d+)').astype(int)

# Merge seeds with game data
df_seeds = df.merge(seeds[['Season', 'TeamID', 'Seed_Num']], 
                    left_on=['Season', 'TeamA_ID'], 
                    right_on=['Season', 'TeamID'], how='left')
df_seeds = df_seeds.rename(columns={'Seed_Num': 'Seed_A'}).drop('TeamID', axis=1)

df_seeds = df_seeds.merge(seeds[['Season', 'TeamID', 'Seed_Num']], 
                          left_on=['Season', 'TeamB_ID'], 
                          right_on=['Season', 'TeamID'], how='left')
df_seeds = df_seeds.rename(columns={'Seed_Num': 'Seed_B'}).drop('TeamID', axis=1)

# Keep only tournament games (with seeds)
tourney = df_seeds.dropna(subset=['Seed_A', 'Seed_B'])

print(f"Tournament games: {len(tourney)}")

In [ ]:
# Determine winner and loser seeds
tourney['Winner_Seed'] = tourney.apply(lambda r: r['Seed_A'] if r['Win_A'] == 1 else r['Seed_B'], axis=1)
tourney['Loser_Seed'] = tourney.apply(lambda r: r['Seed_B'] if r['Win_A'] == 1 else r['Seed_A'], axis=1)

# Create frequency matrix
matchup_matrix = np.zeros((16, 16))
for _, game in tourney.iterrows():
    w = int(game['Winner_Seed']) - 1
    l = int(game['Loser_Seed']) - 1
    matchup_matrix[w, l] += 1

matchup_df = pd.DataFrame(matchup_matrix, index=range(1, 17), columns=range(1, 17))

In [ ]:
# Create heatmap
plt.figure(figsize=(12, 10))
sns.heatmap(matchup_df, annot=True, fmt='.0f', cmap='YlOrRd',
            cbar_kws={'label': 'Number of Games'},
            linewidths=0.5, square=True)
plt.title('Seed Geography: Winner Seed vs Loser Seed', fontsize=14, fontweight='bold')
plt.xlabel('Loser Seed', fontsize=12, fontweight='bold')
plt.ylabel('Winner Seed', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('seed_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Saved: seed_heatmap.png")

**Interpretation:** The heatmap reveals the "Path to the Championship" with dense clusters in expected matchups (1 vs 16, 2 vs 15). High-intensity cells along the top rows show #1 and #2 seeds winning frequently. Notable anomalies appear where lower seeds defeat higher seeds, representing the famous March Madness upsets.

---
## Part D: Upset Autopsy

**Goal:** Analyze the biggest upset using the Four Factors.

In [ ]:
# Find biggest upset (largest seed difference where underdog won)
tourney['Seed_Diff'] = abs(tourney['Seed_A'] - tourney['Seed_B'])
tourney['Is_Upset'] = ((tourney['Win_A'] == 1) & (tourney['Seed_A'] > tourney['Seed_B'])) | \
                      ((tourney['Win_A'] == 0) & (tourney['Seed_A'] < tourney['Seed_B']))

upsets = tourney[tourney['Is_Upset']]
biggest_upset = upsets.loc[upsets['Seed_Diff'].idxmax()]

print(f"Season: {biggest_upset['Season']:.0f}")
print(f"Seed Difference: {biggest_upset['Seed_Diff']:.0f}")
print(f"Team A Seed: {biggest_upset['Seed_A']:.0f}")
print(f"Team B Seed: {biggest_upset['Seed_B']:.0f}")
print(f"Winner: {'Team A' if biggest_upset['Win_A'] == 1 else 'Team B'}")

In [ ]:
# Analyze Four Factors for this game
print("\nFour Factors Analysis:")
print("="*50)
print(f"DIFF_EFG (Shooting):    {biggest_upset['DIFF_EFG']:+.4f}")
print(f"Diff_TOV (Turnovers):   {biggest_upset['Diff_TOV']:+.4f}")
print(f"DIFF_ORB (Rebounding):  {biggest_upset['DIFF_ORB']:+.4f}")
print(f"DIFF_FTR (Free Throws): {biggest_upset['DIFF_FTR']:+.4f}")
print("\nNote: Positive = Team A advantage, Negative = Team B advantage")

In [ ]:
# Visualize the upset
fig, ax = plt.subplots(figsize=(10, 6))

factors = ['eFG%', 'TOV%', 'ORB%', 'FTR']
values = [biggest_upset['DIFF_EFG'], biggest_upset['Diff_TOV'], 
          biggest_upset['DIFF_ORB'], biggest_upset['DIFF_FTR']]

# Adjust if Team B won (flip for interpretation)
if biggest_upset['Win_A'] == 0:
    values = [-v for v in values]

colors = ['green' if v > 0 else 'red' for v in values]
bars = ax.bar(factors, values, color=colors, alpha=0.7, edgecolor='black')

ax.axhline(y=0, color='black', linewidth=1)
ax.set_ylabel('Differential (+ = Underdog Advantage)', fontsize=12, fontweight='bold')
ax.set_title(f'Upset Autopsy: Four Factors Analysis ({biggest_upset["Season"]:.0f})', 
             fontsize=13, fontweight='bold')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('upset_autopsy.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Saved: upset_autopsy.png")

**Interpretation (Write your paragraph here):**

The biggest upset shows [describe which factors favored the underdog]. The underdog won primarily because [mention dominant factor]. Despite the favorite having an advantage in [other factors], these weren't enough to overcome the underdog's superiority in [key winning factor]. This demonstrates that [shooting/rebounding/turnovers] can be decisive even against a heavily favored opponent.

---
## Summary

All visualizations saved:
- correlation_matrix.png
- boxplots_separation.png  
- seed_heatmap.png
- upset_autopsy.png